In [ ]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from collections import Counter
import os, time

In [ ]:
# Ensure 'risky_label' exists before training RF
if "risky_label" not in feature_df.columns:
    spike_threshold = feature_df["spike_count"].quantile(0.9)
    vol_threshold   = feature_df["volatility"].quantile(0.9)
    feature_df["risky_label"] = np.where(
        (feature_df["spike_count"] >= spike_threshold) |
        (feature_df["volatility"]   >= vol_threshold),
        1, 0
    )

In [2]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from collections import Counter
import os, time

X = feature_df[[
    "mean_price_usd",
    "mean_pct_change",
    "volatility",
    "mean_deviation",
    "spike_count"
]].copy()
y = feature_df["risky_label"].values

# Clean data
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
X = np.clip(X, -1e6, 1e6)
X = np.array(X, dtype=float)

# Train/test split
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ------------------------------------------------------------
# 2️⃣ Define Custom Random Forest
# ------------------------------------------------------------
class MyRandomForest:
    def __init__(self, n_trees=10, sample_ratio=0.3, max_depth=None, n_features=None):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.sample_ratio = sample_ratio
        self.n_features = n_features
        self.trees = []

    def bootstrap_sample(self, X, y):
        n_samples = X.shape[0]
        indices = np.random.choice(n_samples, int(n_samples * self.sample_ratio), replace=True)
        return X[indices], y[indices]

    def fit(self, X, y):
        self.trees = []
        n_total_features = X.shape[1]
        for _ in range(self.n_trees):
            # Select random subset of features
            if self.n_features is None:
                selected_features = np.arange(n_total_features)
            else:
                selected_features = np.random.choice(n_total_features, self.n_features, replace=False)
            X_sample, y_sample = self.bootstrap_sample(X[:, selected_features], y)

            tree = DecisionTreeClassifier(max_depth=self.max_depth)
            tree.fit(X_sample, y_sample)
            self.trees.append((tree, selected_features))

    def predict(self, X):
        tree_predictions = []
        for tree, features in self.trees:
            preds = tree.predict(X[:, features])
            tree_predictions.append(preds)
        tree_predictions = np.array(tree_predictions).T
        final_predictions = [Counter(row).most_common(1)[0][0] for row in tree_predictions]
        return np.array(final_predictions)


# ------------------------------------------------------------
# 3️⃣ Train & Evaluate Model
# ------------------------------------------------------------

best_sample = 0.5
best_trees = 15
best_depth = None
num_features = int(x_train.shape[1] * 0.6)

start_time = time.time()
rf_model = MyRandomForest(
    n_trees=best_trees,
    sample_ratio=best_sample,
    max_depth=best_depth,
    n_features=num_features
)
rf_model.fit(x_train, y_train)
training_time = time.time() - start_time

# Evaluate
y_pred = rf_model.predict(x_test)
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("\n🌲 Random Forest Model Evaluation")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"Training Time: {training_time:.2f} seconds")


NameError: name 'feature_df' is not defined